# Семинар 9 - Методы построения оптического потока по последовательности изображений

**Этот семинар содержит оцениваемое домашнее задание**

***

Источник - https://habr.com/ru/post/201406/

$\textbf{Task statement}$: Оптический поток (ОП) – изображение видимого движения, представляющее собой сдвиг каждой точки (пикселя) между двумя изображениями.

По сути, он представляет собой поле скоростей. Суть ОП в том, что для каждой точки изображения $I_{t_0} (\vec{r})$ находится такой вектор сдвига $\delta \vec{r}$, чтобы было соответсвие между исходной точкой и точкой на следущем фрейме $I_{t_1} (\vec{r} + \delta \vec{r})$. В качестве метрики соответвия берут близость интенсивности пикселей, беря во внимание маленькую разницу по времени между кадрами: $\delta{t} = t_{1} - t_{0}$. В более точных методах точку можно привязывать к объекту на основе, например, выделения ключевых точек, а также считать градиенты вокруг точки, лапласианы и проч.

$\textbf{For what}$: Определение собственной скорости, Определение локализации, Улучшение методов трекинга объектов, сегментации, Детектирование событий, Сжатие видеопотока и проч.

![](data/tennis.png)

Разделяют 2 вида оптического потока - плотный (dense) [Farneback method, neural nets], работающий с целым изображением, и выборочный (sparse) [Lucas-Kanade method], работающий с ключевыми точками

In [ ]:
# !wget https://www.bogotobogo.com/python/OpenCV_Python/images/mean_shift_tracking/slow_traffic_small.mp4 -O data/slow_traffic_small.mp4

In [10]:
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

## Lucas-Kanade (sparse)

Пусть $I_{1} = I(x, y, t_{1})$ интенсивность в некоторой точке (x, y) на первом изображении (т. е. в момент времени t). На втором изображении эта точка сдвинулась на (dx, dy), при этом прошло время dt, тогда $I_{2} = I(x + dx, y + dx, t_{1} + dt) \approx I_{1} + I_{x}dx + I_{y}dy +  I_{t}dt$. Из постановки задачи следует, что интенсивность пикселя не изменилась, тогда $I_{1} = I_{2}$. Далее определяем $dx, dy$.

Самое простое решение проблемы – алгоритм Лукаса-Канаде. У нас же на изображении объекты размером больше 1 пикселя, значит, скорее всего, в окрестности текущей точки у других точек будут примерно такие же сдвиги. Поэтому мы возьмем окно вокруг этой точки и минимизируем (по МНК) в нем суммарную погрешность с весовыми коэффициентами, распределенными по Гауссу, то есть так, чтобы наибольший вес имели пиксели, ближе всего находящиеся к исследуемому.

**Полезные материалы:** 
- цикл видео-лекций от First Principles of Computer Vision, посвященный Optical Flow и алгоритму Lucas-Kanade: https://youtube.com/playlist?list=PL2zRqk16wsdoYzrWStffqBAoUY8XdvatV

### Вопрос 1

Перечислите три основных предположения, на которых базируется метод Lucas-Kanade. Почему каждое из них важно для корректной работы алгоритма?

**Ответ:**
1. Малость перемещения: предполагается, что пиксель перемещается между кадрами на небольшое расстояние (обычно не более 1–2 пикселей). Это позволяет линеаризовать изменение яркости с помощью разложения в ряд Тейлора и свести задачу к решению системы линейных уравнений. Без этого предположения метод не сможет корректно аппроксимировать оптический поток.
2. Яркость одной и той же точки объекта не меняется при её движении между соседними кадрами. Если яркость меняется произвольно (например, из-за смены освещения), связь между кадрами теряется, и вычисленный поток будет ошибочным.
3. Соседние пиксели, принадлежащие одному объекту, движутся согласованно (одинаковый вектор перемещения в локальной окрестности). Это позволяет использовать метод наименьших квадратов на окне n×n пикселей для решения недоопределённой системы (одно уравнение на два неизвестных u,v на пиксель).


### Вопрос 2

Объясните, зачем нужен пирамидальный подход в алгоритме Lucas-Kanade. Какую проблему он решает и как именно?

**Ответ:**

Пирамидальный подход расширяет диапазон обнаруживаемых скоростей с ~1 пикселя до десятков и сотен пикселей на кадр, сохраняя точность и сходимость алгоритма, позволяя отслеживать быстрые движения объектов. Исходное изображение многократно сжимается, и получается несколько уровней: самый верхний — очень маленькое изображение (низкое разрешение), самый нижний — исходное. Найденный на верхнем уровне грубый вектор потока используется как начальное приближение для следующего, более детального уровня. На каждом следующем уровне вычисляется лишь небольшая поправка к уже известному вектору. В итоге на исходном разрешении остаётся вычислить только малую остаточную скорость (обычно < 1 пикселя).

### Вопрос 3

С какими проблемами может столкнуться алгоритм Lucas-Kanade при отслеживании точек на видео? Назовите минимум три ограничения.

**Ответ:**
1. Изменение яркости точки между кадрами
2. Предположение о том, что соседние пиксели движутся одинаково, ломается на границах объектов или при перекрытиях
3. Слишком однородная текстура в окне / периодические текстуры (например, клетки на рубашке)


### Задание 1

Напишите реализацию Лукаса-Канаде c помощью numpy и cv2. Сравните с реализацией `cv2.calcOpticalFlowPyrLK`.

In [ ]:
def build_image_pyramid(image, num_levels, scale_factor=0.5):
    """
    Создаёт пирамиду изображений с уменьшающимся разрешением.

    Аргументы:
        image: Исходное изображение (одноканальное, grayscale)
        num_levels: Количество уровней пирамиды
        scale_factor: Коэффициент масштабирования между соседними уровнями

    Возвращает:
        Список изображений [image_level_0, image_level_1, ..., image_level_n-1], где:
        - image_level_0 - исходное изображение
        - Каждый следующий уровень уменьшен относительно предыдущего в scale_factor раз

    Примечание:
        - Используйте cv2.resize с интерполяцией cv2.INTER_LINEAR для уменьшения размера
        - Первым элементом списка должна быть копия исходного изображения
    """
    pyramid = [image.copy()]
    h, w = image.shape[:2]
    for level in range(1, num_levels):
        factor = scale_factor ** level
        new_size = (int(w * factor), int(h * factor))
        pyramid.append(cv2.resize(pyramid[0], new_size, interpolation=cv2.INTER_LINEAR))
    return pyramid



def compute_image_gradients(image):
    """
    Вычисляет пространственные градиенты изображения.

    Аргументы:
        image: Входное изображение (одноканальное, grayscale)

    Возвращает:
        Кортеж (Ix, Iy), где Ix и Iy - градиенты по x и y соответственно

    Примечание:
        - Используйте фильтр Собеля (cv2.Sobel) с ksize=3
        - Используйте тип данных cv2.CV_64F для более точных вычислений
    """
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy


def compute_lk_optical_flow_point(Ix, Iy, It, window_size=5):
    """
    Вычисляет оптический поток по методу Lucas-Kanade для одного окна.

    Аргументы:
        Ix: Градиент изображения по x
        Iy: Градиент изображения по y
        It: Временной градиент (разница между кадрами)
        window_size: Размер окна для вычисления (нечетное число)

    Возвращает:
        Кортеж (u, v) компонентов вектора потока или (None, None) если решение ненадежное

    Примечание:
        - Создайте окно для градиентов, выбрав центральный пиксель и окно размером window_size x window_size
        - Вычислите сумму произведений градиентов для формирования матрицы A:
          A = [[sum(Ix*Ix), sum(Ix*Iy)], [sum(Ix*Iy), sum(Iy*Iy)]]
        - Проверьте обусловленность матрицы A через собственные значения
        - Если минимальное собственное значение меньше порога (например, 1e-4), верните (None, None)
        - Сформируйте вектор b: [-sum(Ix*It), -sum(Iy*It)]
        - Решите систему уравнений A * [u, v] = b
        - Обработайте возможное исключение np.linalg.LinAlgError
    """
    ix2 = Ix * Ix 
    iy2 = Iy * Iy
    ixiy = Ix * Iy

    sum_ix2 = ix2.sum()
    sum_iy2 = iy2.sum()
    sum_ixiy = ixiy.sum()

    A = np.array([[sum_ix2, sum_ixiy], [sum_ixiy, sum_iy2]])
    b = -np.array([Ix.dot(It).sum(), Iy.dot(It).sum()])

    try:
        if np.linalg.cond(A) > 1e4:
            return None, None
        ux, uy = np.linalg.solve(A, b)
        return ux, uy
    except np.linalg.LinAlgError:
        return None, None


def compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=5):
    """
    Вычисляет оптический поток для патча изображения.

    Аргументы:
        prev_patch: Патч из предыдущего кадра
        curr_patch: Соответствующий патч из текущего кадра
        window_size: Размер окна для LK

    Возвращает:
        Кортеж (u, v) компонентов вектора потока для центра патча

    Примечание:
        - Вычислите пространственные градиенты prev_patch с помощью compute_image_gradients
        - Вычислите временной градиент как разность патчей: It = curr_patch - prev_patch
        - Используйте функцию compute_lk_optical_flow_point для вычисления вектора потока
    """

    ix, iy = compute_image_gradients(prev_patch)
    it = curr_patch - prev_patch
    return compute_lk_optical_flow_point(ix, iy, it, window_size)



def track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, point, window_size=15, max_iterations=10, epsilon=0.01):
    """
    Отслеживает точку между кадрами с использованием пирамидального LK.

    Аргументы:
        prev_pyramid: Пирамида предыдущего кадра (список изображений)
        curr_pyramid: Пирамида текущего кадра (список изображений)
        point: Координаты отслеживаемой точки (x, y) на исходном изображении
        window_size: Размер окна для LK
        max_iterations: Максимальное количество итераций для уточнения каждого уровня
        epsilon: Порог для остановки итераций

    Возвращает:
        Кортеж (new_x, new_y) - новые координаты точки на текущем кадре
        или None если отслеживание неуспешно

    Примечание:
        - Начните обработку с верхнего уровня пирамиды (самое маленькое изображение)
        - Масштабируйте исходную точку для соответствия размеру изображения верхнего уровня
        - Для каждого уровня пирамиды (от верхнего к нижнему):
            1. Масштабируйте общее смещение в 2 раза при переходе на уровень ниже
            2. Пересчитайте позицию точки с учетом масштаба текущего уровня
            3. Примените итеративное уточнение позиции с помощью LK:
                a. Проверьте, что точка и окно вокруг неё находятся в границах изображения
                b. Извлеките патчи из предыдущего и текущего кадров
                c. Вычислите смещение с помощью compute_lk_optical_flow_for_patch
                d. Обновите позицию точки
                e. Остановите итерации, если смещение меньше epsilon
            4. Обновите общее смещение
        - Вычислите финальную позицию точки на исходном изображении
    """
    levels = len(prev_pyramid)
    pt = np.array(point, dtype=np.float32)
    flow = np.zeros(2, dtype=np.float32)

    for lvl in reversed(range(levels)):
        scale = 1.0 / (2 ** lvl)
        base_pt = pt * scale + flow * scale
        img_prev = prev_pyramid[lvl]
        img_curr = curr_pyramid[lvl]
        half = window_size // 2
        for _ in range(max_iterations):
            x, y = int(round(base_pt[0])), int(round(base_pt[1]))
            if y - half < 0 or y + half >= img_prev.shape[0] or x - half < 0 or x + half >= img_prev.shape[1]:
                return None
            p_prev = img_prev[y-half:y+half+1, x-half:x+half+1]
            p_curr = img_curr[y-half:y+half+1, x-half:x+half+1]
            if p_prev.shape != (window_size, window_size):
                return None
            delta = compute_lk_optical_flow_for_patch(p_prev, p_curr, window_size)
            if delta[0] is None:
                return None
            base_pt += np.array(delta, dtype=np.float32)
            if abs(delta[0]) < epsilon and abs(delta[1]) < epsilon:
                break
        flow = base_pt - (pt * scale)
    return (pt + flow).tolist()



def lucas_kanade_optical_flow(prev_frame, curr_frame, points,
                             window_size=15, num_pyramid_levels=3,
                             max_iterations=10, epsilon=0.01):
    """
    Вычисляет разреженный оптический поток методом Лукаса-Канаде.

    Аргументы:
        prev_frame: Предыдущий кадр (может быть цветным)
        curr_frame: Текущий кадр (может быть цветным)
        points: Список точек для отслеживания в формате [[x1, y1], [x2, y2], ...]
        window_size: Размер окна для LK
        num_pyramid_levels: Количество уровней в пирамиде изображений
        max_iterations: Максимальное количество итераций на каждом уровне
        epsilon: Порог сходимости для итераций

    Возвращает:
        Кортеж (new_points, status), где:
        - new_points: Массив новых позиций точек в формате [[x1, y1], [x2, y2], ...]
        - status: Массив статусов отслеживания (1 - успешно, 0 - неуспешно)

    Примечание:
        - Преобразуйте входные кадры в полутоновые, если они цветные
        - Нормализуйте изображения к диапазону [0, 1]
        - Создайте пирамиды изображений для обоих кадров
        - Для каждой точки из списка:
            1. Отследите её с помощью track_point_with_pyramid_lk
            2. Сохраните результат в new_points и отметьте статус в status
        - Если отслеживание точки не удалось, установите статус 0 и сохраните исходную точку
    """

    if len(prev_frame.shape) == 3:
            prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    else:
            prev_gray = prev_frame
            
    if len(curr_frame.shape) == 3:
            curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    else:
            curr_gray = curr_frame
        
    prev_gray = prev_gray.astype(np.float32) / 255.0
    curr_gray = curr_gray.astype(np.float32) / 255.0
    
    prev_pyramid = build_image_pyramid(prev_gray, num_pyramid_levels)
    curr_pyramid = build_image_pyramid(curr_gray, num_pyramid_levels)
    
    new_points = np.zeros_like(points, dtype=np.float32)
    status = np.zeros(len(points), dtype=np.int32)

    for i, point in enumerate(points):

        new_point = track_point_with_pyramid_lk(
            prev_pyramid, curr_pyramid, point, 
            window_size, max_iterations, epsilon
        )
        

        if new_point is not None:
            new_points[i] = new_point
            status[i] = 1
        else:
            new_points[i] = point
            status[i] = 0
    
    return new_points, status



def demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)  # Преобразуем в формат [[x1, y1], [x2, y2], ...]

    # Сохраняем изначальные точки для отслеживания через все видео
    initial_points = p0.copy()

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (len(p0), 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью нашей реализации
        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3
        )

        # Выбираем хорошие точки
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0[st == 1] = good_new

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [24]:
result_path = demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4')

  0%|          | 0/913 [00:00<?, ?it/s]

100%|██████████| 913/913 [00:39<00:00, 22.87it/s]

Результат сохранен в output_my_LK.mp4


### Релизация OpenCV - cv2.calcOpticalFlowPyrLK

In [7]:
def demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео с использованием cv2.calcOpticalFlowPyrLK.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Параметры для Lucas-Kanade оптического потока
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (100, 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью встроенной функции cv2.calcOpticalFlowPyrLK
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        # Выбираем хорошие точки
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0 = good_new.reshape(-1, 1, 2)

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [8]:
result_path = demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_LK.mp4')

100%|██████████| 913/913 [00:02<00:00, 351.17it/s]

Результат сохранен в output_opencv_LK.mp4


### Задание 2

В базовой реализации у кода есть одна важная проблема - ключевые точки инициализируются единожды. В реальных задачах необходимо отслеживать точки, которые исчезают из кадра и появляются в других местах. Реализуйте механизм, который будет отслеживать точки, которые пропадают из кадра и добавлять новые точки в те места, где они появляются. Для этого вам нужно будет реализовать механизм поиска новых точек на изображении.

In [ ]:
# your code
import numpy as np
import cv2
from tqdm import tqdm


class DynamicPointTracker:
    
    def __init__(self, 
                 max_points=50,
                 min_distance=10,
                 quality_level=0.01,
                 block_size=3,
                 use_harris=False):
        self.max_points = max_points
        self.min_distance = min_distance
        self.quality_level = quality_level
        self.block_size = block_size
        self.use_harris = use_harris
        
        self.feature_params = dict(
            maxCorners=max_points,
            qualityLevel=quality_level,
            minDistance=min_distance,
            blockSize=block_size,
            useHarrisDetector=use_harris
        )
        self.lk_params = dict(
            winSize=(15, 15),
            maxLevel=3,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
        )
        self.track_history = {}
        self.point_ids = {}
        self.next_id = 0
        
    def add_new_points(self, frame_gray, existing_points, mask=None):
        h, w = frame_gray.shape

        if mask is None:
            mask = np.ones((h, w), dtype=np.uint8) * 255
        else:
            mask = mask.copy()

        radius = self.min_distance
        for pt in existing_points:
            x, y = int(pt[0]), int(pt[1])
            cv2.circle(mask, (x, y), radius, 0, -1)
        
        new_corners = cv2.goodFeaturesToTrack(
            frame_gray, 
            mask=mask, 
            **self.feature_params
        )
        
        if new_corners is None:
            return np.array([]).reshape(0, 2)
        
        new_points = new_corners.reshape(-1, 2)
        
        remaining_slots = self.max_points - len(existing_points)
        if remaining_slots <= 0:
            return np.array([]).reshape(0, 2)
        
        if len(new_points) > remaining_slots:
            new_points = new_points[:remaining_slots]
        
        return new_points
    
    def remove_lost_points(self, points, status, frame_shape, margin=10):
        h, w = frame_shape[:2]
        mask = (status == 1)
        
        if mask.any():
            valid_points = points[mask]
            x_within = (valid_points[:, 0] >= margin) & (valid_points[:, 0] < w - margin)
            y_within = (valid_points[:, 1] >= margin) & (valid_points[:, 1] < h - margin)
            within_frame = x_within & y_within

            temp_mask = np.zeros(len(points), dtype=bool)
            temp_mask[mask] = within_frame
            mask = temp_mask
        
        return mask
    
    def update_track_history(self, points, status, frame_idx):
        # Удаляем потерянные точки из истории
        lost_points = []
        for pid, (pt, last_frame) in self.track_history.items():
            if last_frame < frame_idx - 30:  # Если точка не обновлялась 30 кадров
                lost_points.append(pid)
        
        for pid in lost_points:
            del self.track_history[pid]
            if pid in self.point_ids:
                del self.point_ids[pid]
        
        # Обновляем существующие точки
        for i, (pt, s) in enumerate(zip(points, status)):
            if s == 1:
                if i in self.point_ids:
                    pid = self.point_ids[i]
                    self.track_history[pid] = (pt.copy(), frame_idx)
    
    def track(self, old_gray, frame_gray, points, status=None):
        if len(points) == 0:
            return np.array([]).reshape(0, 2), np.array([])
        
        # Преобразуем точки в формат OpenCV (N x 1 x 2)
        pts = points.reshape(-1, 1, 2).astype(np.float32)
        
        new_pts, st, err = cv2.calcOpticalFlowPyrLK(
            old_gray, frame_gray, pts, None, **self.lk_params
        )
        
        if new_pts is None:
            return points, np.zeros(len(points), dtype=np.uint8)
        
        new_pts = new_pts.reshape(-1, 2)
        st = st.reshape(-1)
        
        return new_pts, st


def dynamic_optical_flow_demo(video_path='data/slow_traffic_small.mp4', 
                              output_path='output_dynamic_LK.mp4',
                              max_points=50,
                              min_distance=15,
                              quality_level=0.05,
                              add_points_every_n_frames=5):

    cap = cv2.VideoCapture(video_path)
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    tracker = DynamicPointTracker(
        max_points=max_points,
        min_distance=min_distance,
        quality_level=quality_level
    )
    
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    
    initial_points = cv2.goodFeaturesToTrack(old_gray, mask=None, **tracker.feature_params)
    if initial_points is not None:
        points = initial_points.reshape(-1, 2)
        tracker.point_ids = {i: tracker.next_id + i for i in range(len(points))}
        tracker.next_id += len(points)
        for i, pt in enumerate(points):
            tracker.track_history[tracker.point_ids[i]] = (pt.copy(), 0)
    else:
        points = np.array([]).reshape(0, 2)
    
    mask = np.zeros_like(old_frame)
    color = np.random.randint(0, 255, (max_points, 3))
    frames_since_last_add = 0
    
    for frame_idx in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            break
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # 1. Отслеживаем существующие точки
        if len(points) > 0:
            new_points, status = tracker.track(old_gray, frame_gray, points)
            
            # 2. Удаляем потерянные точки
            keep_mask = tracker.remove_lost_points(new_points, status, frame.shape)
            points = new_points[keep_mask]
            status = status[keep_mask]
            new_point_ids = {}
            for i, (pt, s) in enumerate(zip(points, status)):
                original_idx = np.where(keep_mask)[0][i]
                if original_idx in tracker.point_ids:
                    new_point_ids[i] = tracker.point_ids[original_idx]
            tracker.point_ids = new_point_ids
        else:
            status = np.array([])
        
        # 3. Добавляем новые точки
        frames_since_last_add += 1
        should_add = (frames_since_last_add >= add_points_every_n_frames) or (len(points) < max_points // 2)
        if should_add and len(points) < max_points:
            new_candidates = tracker.add_new_points(frame_gray, points)
            if len(new_candidates) > 0:
                start_id = tracker.next_id
                for i in range(len(new_candidates)):
                    tracker.point_ids[len(points) + i] = start_id + i
                    tracker.track_history[start_id + i] = (new_candidates[i].copy(), frame_idx)
                tracker.next_id += len(new_candidates)
                points = np.vstack([points, new_candidates]) if len(points) > 0 else new_candidates
                status = np.concatenate([status, np.ones(len(new_candidates), dtype=np.uint8)])
            frames_since_last_add = 0
        
        # 4. Визуализация
        for i, (pt, s) in enumerate(zip(points, status)):
            if s == 1 and i in tracker.point_ids:
                pid = tracker.point_ids[i]
                if pid in tracker.track_history:
                    prev_pt, _ = tracker.track_history[pid]
                    mask = cv2.line(mask, 
                                   (int(pt[0]), int(pt[1])), 
                                   (int(prev_pt[0]), int(prev_pt[1])), 
                                   color[pid % max_points].tolist(), 
                                   2)
                    tracker.track_history[pid] = (pt.copy(), frame_idx)
                frame = cv2.circle(frame, 
                                  (int(pt[0]), int(pt[1])), 
                                  4, 
                                  color[pid % max_points].tolist(), 
                                  -1)

        img = cv2.add(frame, mask)
        
        info_text = f"Points: {len(points)}/{max_points} | Frame: {frame_idx}"
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                   0.7, (255, 255, 255), 2)
        
        out.write(img)
        old_gray = frame_gray.copy()

    cap.release()
    out.release()
    
    print(f"Результат сохранен в {output_path}")
    return output_path



dynamic_optical_flow_demo(
    video_path='data/slow_traffic_small.mp4',
    output_path='output_dynamic_LK.mp4',
    max_points=50,
    min_distance=15,
    quality_level=0.05,
    add_points_every_n_frames=3
)

100%|██████████| 913/913 [00:03<00:00, 262.89it/s]


Результат сохранен в output_dynamic_LK.mp4


'output_dynamic_LK.mp4'

### Вопрос 4

В чем основное отличие разреженного (sparse) оптического потока Lucas-Kanade от плотного (dense) оптического потока (например, метода Farneback)?

**Ответ:**

Основное отличие заключается в объёме вычисляемой информации и подходе к выбору точек.
- Разреженный метод вычисляет перемещение только для заранее выбранного набора характерных точек.
- Плотный метод вычисляет вектор движения для каждого пикселя изображения (полное поле оптического потока).

## Farneback (dense)

Метод Farneback носит несколько более глобальный характер, чем метод Лукаса-Канаде. Он опирается на предположение о том, что на всем изображении оптический поток будет достаточно гладким.

# Вопрос 5

Перечислите основные шаги алгоритма Farneback для расчета оптического потока.

**Ответ:**
1. Полиномиальная аппроксимация каждого кадра
2. Представление полиномов в канонической форме
3. Предположение о глобальном сдвиге
4. Вычисление вектора сдвига для каждого пикселя
5. Итеративное уточнение
6. Пирамидальная обработка (для больших смещений)

### Вопрос 6

Каким образом в методе Farneback обрабатываются большие смещения объектов между кадрами?

**Ответ:**

В методе Farneback большие смещения объектов обрабатываются с помощью комбинации двух ключевых механизмов: многоуровневой пирамиды изображений и итеративного уточнения (warping).

Пирамидальная обработка изображений:
1. Сначала поток вычисляется на самом грубом уровне (маленькое изображение).
2. Результат проецируется на следующий уровень как начальное приближение.
3. На каждом уровне уточняется остаточный поток.

Итеративное уточнение:
1. Вычисляется грубый поток.
2. Второй кадр пересэмплируется с компенсацией этого потока (warping).
3. Вычисляется остаточный поток на новом изображении.
4. Процесс повторяется (обычно 5–10 итераций) до сходимости.


In [ ]:
def demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_Farneback.mp4'):
    """
    Демонстрация работы алгоритма плотного оптического потока Farneback на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Берем первый кадр и преобразуем его в оттенки серого
    ret, frame1 = cap.read()
    if not ret:
        print('Не удалось прочитать видео')
        return None

    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    # Создаем HSV-изображение для визуализации потока
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255  # Насыщенность устанавливаем на максимум

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame2 = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток методом Farneback
        # Параметры:
        # - 0.5: коэффициент масштабирования для пирамиды изображений
        # - 3: кол-во уровней пирамиды
        # - 15: размер окна для усреднения
        # - 3: число итераций на каждом уровне пирамиды
        # - 5: размер окна для полиномиальной аппроксимации
        # - 1.2: стандартное отклонение для сглаживания
        flow = cv2.calcOpticalFlowFarneback(
            prvs, next_frame, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # Преобразуем векторы потока из декартовых координат в полярные
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Кодируем направление потока как оттенок (hue)
        hsv[..., 0] = ang * 180 / np.pi / 2

        # Кодируем величину потока как яркость (value)
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        # Преобразуем HSV в BGR для отображения
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # Записываем результат
        out.write(bgr)

        # Обновляем предыдущий кадр
        prvs = next_frame

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [ ]:
result_path = demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_farneback.mp4')

### Вопрос 7

Как влияет предварительная обработка изображений (фильтрация шума, выравнивание гистограмм) на качество оптического потока, получаемого методом Farneback? Предложите оптимальный пайплайн предобработки.

**Ответ:**

Предварительная обработка изображений критически влияет на качество оптического потока в методе Farneback, так как алгоритм опирается на вычисление локальных полиномиальных аппроксимаций, а значит — на градиенты и локальную структуру изображения. Шум и неравномерная освещённость могут серьёзно ухудшить результат.

Оптимальный пайплайн предобработки:
1. Конвертация в градации серого
2. Слабый гауссовский blur (подавление шума)
3. Адаптивная эквализация гистограммы (CLAHE)
4. Приведение к типу float32 (для Farneback)